# 03: Data Loaders - MLM & CLM

In [1]:
!uv pip install --quiet tokenizers datasets transformers torch

## Dataset & Tokenizer

In [2]:
from datasets import Dataset

dataset = Dataset.from_dict({
    "text": [
        "I play cricket.",
        "Yesterday, I was about to hit him.",
        "Is this even considered?",
        "You can't get my mobile number.",
    ]
})

In [3]:
from tokenizers import Tokenizer
from tokenizers.models import BPE


SPECIAL_TOKENS = [
    "[_UNK_]",  # unknown token
    "[_PAD_]",  # padding token
    "[_INIT_]", # init token
    "[_STOP_]"  # stop token
]

tokenizer = Tokenizer(BPE(unk_token="[_UNK_]"))

In [4]:
from tokenizers.trainers import BpeTrainer

trainer = BpeTrainer(special_tokens=SPECIAL_TOKENS)

In [5]:
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder

tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)
tokenizer.decoder = ByteLevelDecoder()

In [6]:
tokenizer.train_from_iterator(dataset["text"], trainer)

In [7]:
for sentence in dataset["text"]:
    ids = tokenizer.encode(sentence).ids
    decoded = tokenizer.decode(ids)
    print(f"Original : {sentence}")
    print(f"IDs.     : {ids}")
    print(f"Decoded  : {decoded}")
    print()

Original : I play cricket.
IDs.     : [8, 75, 95, 6]
Decoded  : I play cricket.

Original : Yesterday, I was about to hit him.
IDs.     : [97, 5, 70, 76, 91, 85, 88, 87, 6]
Decoded  : Yesterday, I was about to hit him.

Original : Is this even considered?
IDs.     : [42, 86, 92, 94, 7]
Decoded  : Is this even considered?

Original : You can't get my mobile number.
IDs.     : [44, 79, 41, 73, 83, 96, 93, 6]
Decoded  : You can't get my mobile number.



## 1. Masked Language Modeling (MLM)

- Used by BERT-style models.
- `DataCollatorForLanguageModeling` randomly masks 15% of tokens and the model learns to predict them.

In [12]:
from transformers import PreTrainedTokenizerFast, DataCollatorForLanguageModeling
from torch.utils.data import DataLoader

# Wrap the tokenizers.Tokenizer for HuggingFace compatibility
fast_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer,
    unk_token="[_UNK_]",
    pad_token="[_PAD_]",
    bos_token="[_INIT_]",
    eos_token="[_STOP_]",
)
fast_tokenizer.add_special_tokens({"mask_token": "[MASK]"})

# Tokenize dataset
def tokenize_fn(batch):
    return fast_tokenizer(batch["text"], truncation=True)

tokenized_dataset = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
tokenized_dataset.set_format("torch")

# MLM collator + DataLoader
mlm_collator = DataCollatorForLanguageModeling(tokenizer=fast_tokenizer, mlm=True, mlm_probability=0.15)
mlm_loader = DataLoader(tokenized_dataset, batch_size=4, collate_fn=mlm_collator)

# Inspect one batch
batch = next(iter(mlm_loader))
for i in range(len(dataset["text"])):
    input_tokens = fast_tokenizer.convert_ids_to_tokens(batch["input_ids"][i])
    label_tokens = [
        fast_tokenizer.convert_ids_to_tokens([lbl.item()])[0] if lbl != -100 else "[X]"
        for lbl in batch["labels"][i]
    ]
    print(f"Sample {i}")
    print(f"  Input  : {input_tokens}")
    print(f"  Labels : {label_tokens}")
    print()

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Sample 0
  Input  : ['I', 'Ġplay', '[MASK]', '.', '[_PAD_]', '[_PAD_]', '[_PAD_]', '[_PAD_]', '[_PAD_]']
  Labels : ['[X]', '[X]', 'Ġcricket', '[X]', '[X]', '[X]', '[X]', '[X]', '[X]']

Sample 1
  Input  : ['Yesterday', ',', 'ĠI', 'Ġwas', 'Ġabout', 'Ġto', 'Ġhit', 'Ġhim', '.']
  Labels : ['[X]', '[X]', '[X]', 'Ġwas', '[X]', '[X]', '[X]', '[X]', '[X]']

Sample 2
  Input  : ['Is', 'Ġthis', 'Ġeven', 'Ġconsidered', '?', '[_PAD_]', '[_PAD_]', '[_PAD_]', '[_PAD_]']
  Labels : ['[X]', '[X]', '[X]', '[X]', '[X]', '[X]', '[X]', '[X]', '[X]']

Sample 3
  Input  : ['You', 'Ġcan', "'t", 'Ġget', 'Ġmy', 'Ġmobile', '[MASK]', '.', '[_PAD_]']
  Labels : ['[X]', '[X]', '[X]', '[X]', '[X]', '[X]', 'Ġnumber', '[X]', '[X]']



### What you're seeing

Take sample 0 as an example:

```
Position : [    0,         1,          2,       3,        4,        5,        6,        7,        8   ]
Input    : [  'I',  '[MASK]', 'Ġcricket',     '.',  '[PAD]',  '[PAD]',  '[PAD]',  '[PAD]',   '[PAD]'  ]
Labels   : ['[X]',   'Ġplay',      '[X]',   '[X]',    '[X]',    '[X]',    '[X]',    '[X]',     '[X]'  ]
```

**Step by step:**

1. **Original sentence:** `I play cricket.`  — tokenizes to `['I', 'Ġplay', 'Ġcricket', '.']`

2. **Padding:** The other sentence in this batch is longer, so this one is padded to match:
   `['I', 'Ġplay', 'Ġcricket', '.', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']`

3. **Masking (15% chance per token):** The collator randomly picked position 1 (`Ġplay`) to mask.  
   - Position 1 in `input` becomes `[MASK]`  
   - Position 1 in `labels` becomes `Ġplay` (the original token — the answer)  
   - All other positions in `labels` become `-100` (shown as `[X]`) — **ignored in loss**

4. **During training**, the model:
   - Reads the full input: `['I', '[MASK]', 'Ġcricket', '.', '[PAD]', ...]`
   - Produces a probability distribution over the vocabulary at **every** position
   - Loss is computed **only at position 1**, comparing the model's prediction to `Ġplay`
   - The model uses context from both sides (`I` on the left, `Ġcricket .` on the right) to guess the masked word — this is the **bidirectional** nature of MLM

**Why `-100`?**  
PyTorch's `CrossEntropyLoss` ignores any position where the label is `-100` by default. This is how the collator tells the loss function: *"only penalise the model for the masked positions, not the rest."*

## 2. Causal Language Modeling (CLM)

- Used by GPT-style models.
- `DataCollatorForLanguageModeling` with `mlm=False` handles dataloading for CLM.
- Dataloader does NOT give shifted outputs. The model handles that.

In [9]:
# CLM collator + DataLoader (reuses fast_tokenizer and tokenized_dataset from above)
clm_collator = DataCollatorForLanguageModeling(tokenizer=fast_tokenizer, mlm=False)
clm_loader = DataLoader(tokenized_dataset, batch_size=4, collate_fn=clm_collator)

# Inspect one batch
batch = next(iter(clm_loader))
for i in range(len(dataset["text"])):
    input_tokens = fast_tokenizer.convert_ids_to_tokens(batch["input_ids"][i])
    label_tokens = [
        fast_tokenizer.convert_ids_to_tokens([lbl.item()])[0] if lbl != -100 else "[X]"
        for lbl in batch["labels"][i]
    ]
    print(f"Sample {i}")
    print(f"  Input  : {input_tokens}")
    print(f"  Labels : {label_tokens}")
    print()

Sample 0
  Input  : ['I', 'Ġplay', 'Ġcricket', '.', '[_PAD_]', '[_PAD_]', '[_PAD_]', '[_PAD_]', '[_PAD_]']
  Labels : ['I', 'Ġplay', 'Ġcricket', '.', '[X]', '[X]', '[X]', '[X]', '[X]']

Sample 1
  Input  : ['Yesterday', ',', 'ĠI', 'Ġwas', 'Ġabout', 'Ġto', 'Ġhit', 'Ġhim', '.']
  Labels : ['Yesterday', ',', 'ĠI', 'Ġwas', 'Ġabout', 'Ġto', 'Ġhit', 'Ġhim', '.']

Sample 2
  Input  : ['Is', 'Ġthis', 'Ġeven', 'Ġconsidered', '?', '[_PAD_]', '[_PAD_]', '[_PAD_]', '[_PAD_]']
  Labels : ['Is', 'Ġthis', 'Ġeven', 'Ġconsidered', '?', '[X]', '[X]', '[X]', '[X]']

Sample 3
  Input  : ['You', 'Ġcan', "'t", 'Ġget', 'Ġmy', 'Ġmobile', 'Ġnumber', '.', '[_PAD_]']
  Labels : ['You', 'Ġcan', "'t", 'Ġget', 'Ġmy', 'Ġmobile', 'Ġnumber', '.', '[X]']



### What you're seeing

Take sample 0 (`I play cricket.`) as an example:

```
Position : [ 0      1          2          3      4       5       6       7       8    ]
Input    : ['I', 'Ġplay', 'Ġcricket',   '.',  '[PAD]','[PAD]','[PAD]','[PAD]','[PAD]']
Labels   : ['Ġplay','Ġcricket',  '.',  '[PAD]',  '[X]',    '[X]',    '[X]',    '[X]',   '[X]' ]
```

**Step by step:**

1. **Original sentence:** `I play cricket.` — tokenizes to `['I', 'Ġplay', 'Ġcricket', '.']`

2. **Padding:** Shorter sentence is padded to match the longest in the batch:
   `['I', 'Ġplay', 'Ġcricket', '.', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']`

3. **Labels = input shifted left by one:**  
   - Position 0 label = `Ġplay` → *"given `I`, predict `play`"*  
   - Position 1 label = `Ġcricket` → *"given `I play`, predict `cricket`"*  
   - Position 2 label = `.` → *"given `I play cricket`, predict `.`"*  
   - Position 3 label = `[PAD]` → first pad token  
   - Positions 4–8 label = `[X]` (`-100`) → pad-to-pad, ignored in loss

4. **During training**, the model:
   - Reads each token with a **causal mask** — position `t` can only attend to positions `0..t` (no peeking right)
   - At every position `t`, predicts what token comes next
   - Loss is computed at every **non-padding** position — unlike MLM where only masked positions contribute

**MLM vs CLM at a glance:**

| | MLM (BERT-style) | CLM (GPT-style) |
|---|---|---|
| Attention | Bidirectional (sees full sequence) | Unidirectional (left-to-right only) |
| Labels | Only masked positions (~15%) | Every non-padding position |
| Goal | Fill in blanks | Predict next token |
| Good for | Understanding tasks (classification, QA) | Generation tasks (chat, completion) |